## **Lectura 1: Símplex para problemas de flujo en redes**


**Antes de empezar**

Les recomiendo ver este video del algoritmo [SIMPLEX](https://www.youtube.com/watch?v=E72DWgKP_1Y).

Otra curiosidad, a que no se imaginan que el SIMPLEX es una figura geométrica, [aquí](https://galileo-unbound.blog/2023/05/03/the-mighty-simplex/) les dejo un link que lo explica.

El **problema de flujo de costo mínimo** (FCM) consiste en encontrar la asignación de flujos sobre los arcos de una red que minimiza el costo total de transporte, sujeto a restricciones de balance de flujo en cada nodo. Formalmente, dada una red dirigida $G = (N, A)$ con $|N| = n$ nodos y $|A| = m$ arcos, el problema se escribe:

$$\min \sum_{(i,j) \in A} c_{ij}\, x_{ij}$$

$$\text{s.a.} \quad \sum_{j \,:\, (i,j) \in A} x_{ij} \;-\; \sum_{j \,:\, (j,i) \in A} x_{ji} = b_i \qquad \forall\, i \in N$$

$$x_{ij} \geq 0 \qquad \forall\, (i,j) \in A$$

donde $c_{ij}$ es el costo por unidad de flujo en el arco $(i,j)$, $x_{ij}$ es el flujo en dicho arco, y $b_i$ es el balance neto del nodo $i$ (positivo si es un nodo fuente, negativo si es un nodo destino, cero si es un nodo de transbordo).

Para facilitar el análisis, en esta lectura se asume que:

- La demanda de la red se encuentra balanceada, es decir, $\sum_{i \in N} b_i = 0$.
- No existe restricción de capacidad en los arcos (equivalente a $u_{ij} = +\infty$).
- Solo se exige que el flujo sea no negativo ($x_{ij} \geq 0$).


### Solución mediante el método símplex

Dado que el problema de flujo de costo mínimo es un programa lineal, puede resolverse con el método símplex estándar. Sin embargo, aplicarlo directamente presenta dificultades prácticas:

- El número de variables y restricciones crece con el tamaño de la red. Para una red con $n$ nodos y $m$ arcos, la matriz de restricciones tiene dimensiones $n \times m$. Redes de aplicación real (como la red de 417 nodos del proyecto del curso) generan matrices muy grandes.
- En cada iteración del símplex se debe factorizar o actualizar la matriz de base $B$, operación de costo $O(n^3)$ en el caso general.
- Las soluciones del problema de flujo suelen ser dispersas (muchos arcos con flujo cero), por lo que la representación densa de la matriz desperdicia memoria y tiempo de cómputo.

A pesar de estas dificultades, la **estructura especial** de la matriz de restricciones del FCM permite desarrollar una variante altamente eficiente del símplex, conocida como el **símplex de redes**, en la que las operaciones de actualización de base se reducen a manipulaciones sobre árboles.


### Rango de la matriz de restricciones

Las restricciones de balance de flujo para cada nodo $i \in N$ son:

$$\sum_{j \,:\, (i,j) \in A} x_{ij} \;-\; \sum_{j \,:\, (j,i) \in A} x_{ji} = b_i \qquad \forall\, i \in N$$

¿Qué ocurre si sumamos estas $n$ ecuaciones sobre todos los nodos?

- Cada arco $(i,j)$ aparece exactamente dos veces: con signo positivo en la ecuación del nodo $i$ y con signo negativo en la ecuación del nodo $j$.
- Los términos se cancelan par a par, de modo que la suma de todas las ecuaciones es $0 = \sum_{i \in N} b_i = 0$.

Esto significa que cualquiera de las $n$ ecuaciones puede obtenerse como combinación lineal de las restantes $n-1$: el sistema de ecuaciones es **linealmente dependiente**. En consecuencia:

$$\text{rango}(A) = n - 1$$

donde $A$ es la matriz de incidencia nodo-arco. Para aplicar el símplex es necesario eliminar una ecuación redundante (se puede elegir cualquier nodo de referencia), obteniendo un sistema de $n-1$ ecuaciones linealmente independientes.

![](../Images/simplex3.png)

![](../Images/simplex4.png)


### Repaso del método símplex

Considere el siguiente programa lineal en forma estándar:

$$\min \;\{ c^T x \mid Ax = b,\; x \geq 0 \}$$


En este problema, $A$ es una matriz de rango máximo de dimensiones $m \times n$ con $m < n$. Por tanto, el sistema $Ax = b$ tiene infinitas soluciones. Una **solución básica** se obtiene seleccionando $m$ columnas de $A$ que formen una submatriz cuadrada no singular $B$ (la **base**), asignando $x_N = 0$ a las $n - m$ variables no básicas, y resolviendo $x_B = B^{-1}b$. Si además $x_B \geq 0$, la solución básica es **factible**.

Separando las variables básicas $x_B$ de las no básicas $x_N$, el sistema de restricciones se puede escribir como:

$$Bx_B + Nx_N = b$$

En el contexto del problema de flujo de costo mínimo, donde la matriz de restricciones tiene rango $n-1$, una solución básica poseerá a lo más $n - 1$ variables básicas (arcos con flujo determinado por la base) y $m - (n-1)$ variables no básicas (arcos fijados en cero).


## Solución básica en una red

Una **solución básica factible** del problema de flujo de costo mínimo corresponde a un **árbol generador** (spanning tree) de la red. En efecto:

- La base $B$ está formada por $n-1$ columnas de la matriz de incidencia nodo-arco.
- Puede demostrarse que un conjunto de $n-1$ columnas de la matriz de incidencia forma una base no singular si y solo si los arcos correspondientes forman un árbol generador de la red (Ahuja et al., 1993, Cap. 7).
- Los flujos en los arcos del árbol se calculan resolviendo el sistema triangular $Bx_B = b$, mientras que los arcos fuera del árbol tienen flujo cero.

Esta equivalencia entre bases y árboles es la clave de la eficiencia del símplex de redes: en lugar de factorizar una matriz de $n \times n$, basta con recorrer el árbol para calcular flujos y potenciales.

**Ejemplo**

![](../Images/simplex1.png)

¿Qué representa una solución básica en esta red?

![](../Images/simplex2.png)


### Esquema del algoritmo símplex de redes

Con los elementos anteriores, el símplex de redes opera sobre árboles generadores factibles:

1. Encontrar un árbol generador inicial factible $T^{(0)}$.
2. Calcular los potenciales $\pi_i$ y los costos reducidos $\bar{c}_{ij}$ de todos los arcos.
3. Si $\bar{c}_{ij} \geq 0$ para todo arco $(i,j) \notin T$, la solución actual es **óptima**. Detenerse.
4. Seleccionar un arco $(p,q) \notin T$ con $\bar{c}_{pq} < 0$ (**arco entrante**).
5. Agregar $(p,q)$ al árbol: se forma un único ciclo $C$ en $T \cup \{(p,q)\}$.
6. Aumentar el flujo en $\Delta$ unidades a lo largo del ciclo $C$, determinando el **arco saliente** como el primero que alcanza flujo cero.
7. Actualizar el árbol, los flujos, los potenciales y los costos reducidos. Ir al paso 3.

¿Cómo se calculan eficientemente los potenciales y los costos reducidos? ¿Cómo se elige $\Delta$? Estos aspectos se desarrollan en las secciones siguientes.


### Costos reducidos

Consideremos la descomposición del programa lineal en variables básicas $x_B$ y no básicas $x_N$:

$$\min \quad c_B^T x_B + c_N^T x_N$$

$$\text{s.a.} \quad Bx_B + Nx_N = b$$

$$x_B \geq 0, \quad x_N \geq 0$$


Una base corresponde al óptimo del problema si su solución asociada es factible ($x_B = B^{-1}b \geq 0$) y si los costos reducidos de todas las variables no básicas son no negativos.

**Derivación de los costos reducidos.** Despejando $x_B$ de $Bx_B + Nx_N = b$:

$$x_B = B^{-1}b - B^{-1}Nx_N$$

Sustituyendo en la función objetivo:

$$z = c_B^T\bigl(B^{-1}b - B^{-1}Nx_N\bigr) + c_N^T x_N = c_B^T B^{-1}b + (c_N^T - c_B^T B^{-1}N)\,x_N$$

En la solución básica actual se tiene $x_N = 0$, por lo que $z = c_B^T B^{-1}b$. Para que ningún cambio en $x_N$ mejore el objetivo, se requiere que el coeficiente de cada $x_{N_j}$ sea no negativo:

$$\bar{c}_j = c_j - c_B^T B^{-1} N_j \;\geq\; 0 \qquad \forall\, j \in N$$

Si existe alguna variable no básica con $\bar{c}_j < 0$, introducirla en la base mejora el objetivo.


Definiendo el **vector de potenciales** (variables duales) como $\pi^T = c_B^T B^{-1}$, la expresión anterior se reescribe como:

$$\bar{c}_j = c_j - \pi^T N_j \qquad \forall\, j \notin B$$

Los valores del vector $\pi$ reciben el nombre de **potenciales nodales** y tienen una interpretación económica precisa: $\pi_i$ representa el precio sombra del nodo $i$, es decir, cuánto varía el costo óptimo ante un incremento unitario en $b_i$.


### Problema dual

Sea el problema primal:

$$\min \;\{ c^T x \mid Ax = b,\; x \geq 0 \}$$

El problema dual asociado es:

$$\max \;\{ b^T \pi \mid A^T \pi \leq c \}$$

Las principales relaciones de dualidad relevantes para el símplex de redes son:

- **Dualidad débil:** Si $x$ es factible para el primal y $\pi$ es factible para el dual, entonces $c^T x \geq b^T \pi$.
- **Dualidad fuerte:** En el óptimo, $c^T x^* = b^T \pi^*$.
- Las variables duales $\pi_i$ son irrestrictas en signo porque las restricciones del primal son igualdades.
- **Holgura complementaria:** $x_j(c_j - \pi^T A_j) = 0$ para toda variable $j$.

![](../Images/Duality_overview.png)


#### Problema dual del problema de flujo de costo mínimo

El problema de flujo de costo mínimo se puede escribir en forma matricial como:

$$\min \quad Z = \sum_{(i,j) \in A} c_{ij}\, x_{ij}$$

$$\sum_{j \,:\, (i,j) \in A} x_{ij} - \sum_{j \,:\, (j,i) \in A} x_{ji} = b_i \qquad \forall\, i \in N$$

$$x_{ij} \geq 0 \qquad \forall\, (i,j) \in A$$

El problema dual asociado, con variables $\pi_i$ para cada nodo $i \in N$, es:

$$\max \quad Z_D = \sum_{i \in N} b_i\, \pi_i$$

$$\pi_i - \pi_j \leq c_{ij} \qquad \forall\, (i,j) \in A$$

$$\pi_i \;\text{irrestricta} \qquad \forall\, i \in N$$

Observaciones importantes:

- Como las restricciones del primal son igualdades, las variables duales $\pi_i$ son irrestrictas en signo.
- Una de las variables $\pi_i$ queda indeterminada (el sistema dual tiene una grado de libertad), por lo que se fija arbitrariamente $\pi_r = 0$ para algún nodo raíz $r$.
- Por holgura complementaria, si $x_{ij} > 0$ entonces la restricción dual es activa: $\pi_i - \pi_j = c_{ij}$. Esto permite calcular todos los potenciales recorriendo el árbol a partir del nodo raíz.


![](../Images/simplex5.png)

![](../Images/simplex6.png)

### Costos reducidos en el problema de flujo de costo mínimo

En el símplex de redes, los costos reducidos adoptan una forma especialmente simple. Dado que la columna de la matriz de incidencia correspondiente al arco $(i,j)$ tiene un $+1$ en la fila $i$ y un $-1$ en la fila $j$, el costo reducido de $x_{ij}$ es:

$$\bar{c}_{ij} = c_{ij} - (\pi_i - \pi_j)$$

Con la convención de potenciales que se detalla en la sección siguiente, esta expresión se convierte en:

$$\bar{c}_{ij} = c_{ij} + \pi_i - \pi_j$$

La condición de optimalidad $\bar{c}_{ij} \geq 0$ para todo arco fuera de la base equivale, entonces, a verificar que ningún arco no básico viole la restricción dual $\pi_i - \pi_j \leq c_{ij}$.


**Recapitulación**

Los elementos centrales del símplex de redes son:

1. El problema de flujo de costo mínimo es un programa lineal cuya estructura de red permite resolverlo de manera especialmente eficiente.
2. Las variables básicas de una solución básica factible corresponden a los arcos de un **árbol generador** de la red.
3. Para cada arco $(i,j)$ básico se cumple $\bar{c}_{ij} = 0$, lo que equivale a la relación de potenciales $\pi_j = c_{ij} + \pi_i$.
4. Para los arcos $(i,j)$ no básicos, el costo reducido es $\bar{c}_{ij} = c_{ij} + \pi_i - \pi_j$.
5. La **condición de optimalidad** es $\bar{c}_{ij} \geq 0$ para todo $(i,j) \notin T$. Si algún arco no básico tiene costo reducido negativo, introducirlo en la base reduce el costo total.


## Símplex de redes: algoritmo detallado

El algoritmo símplex de redes es una variante del símplex que se mueve entre **árboles generadores factibles**. En cada iteración se realiza un intercambio de arcos en el árbol (pivote), que corresponde a un cambio de base en el símplex estándar. Los pasos del algoritmo son:

1. Encontrar un árbol generador inicial factible $T$.
2. Determinar si el árbol es óptimo calculando potenciales $\pi_i$ y costos reducidos $\bar{c}_{ij}$. Si $\bar{c}_{ij} \geq 0$ para todo $(i,j) \notin T$, detenerse.
3. Seleccionar un arco no básico $(p,q)$ con $\bar{c}_{pq} < 0$ (arco entrante).
4. Agregar $(p,q)$ al árbol y determinar el ciclo $C$ que se forma.
5. Determinar el arco $(k,l)$ que sale del árbol: es el arco del ciclo con menor flujo en la dirección en que el ciclo aumenta el flujo. El flujo se incrementa en $\Delta = \min_{(i,j) \in C^-} x_{ij}$, donde $C^-$ es el conjunto de arcos del ciclo cuyo flujo debe disminuir.
6. Actualizar el árbol $T$, los flujos $x_{ij}$, los potenciales $\pi_i$ y los costos reducidos $\bar{c}_{ij}$. Ir al paso 2.


De manera precisa, el algoritmo opera como sigue:

1. **Inicialización:** Encontrar un árbol generador factible inicial $T^{(1)}$. Calcular los flujos asociados $x_{ij}$, los potenciales $\pi_i$ y los costos reducidos $\bar{c}_{ij}$.

2. **Iteración $k$:** Mientras existan arcos fuera de $T^{(k)}$ que no cumplan la condición de optimalidad:
   - Seleccionar un arco $(p,q) \notin T^{(k)}$ con $\bar{c}_{pq} < 0$.
   - Agregar $(p,q)$ a $T^{(k)}$ y determinar el único ciclo $C$ que se forma.
   - Calcular $\Delta = \min_{(i,j) \in C^-} x_{ij}$, donde $C^-$ es el conjunto de arcos de $C$ en sentido contrario al de $(p,q)$.
   - Actualizar los flujos: aumentar en $\Delta$ los arcos de $C$ en el sentido de $(p,q)$ y disminuir en $\Delta$ los de $C^-$.
   - El arco $(k,l) \in C^-$ que alcanza flujo cero sale de la base.
   - Llamar $T^{(k+1)}$ al nuevo árbol. Actualizar potenciales y costos reducidos.


### Convención sobre los potenciales

Para facilitar los cálculos, en el símplex de redes se trabaja con el **inverso aditivo** de los potenciales duales, es decir, con $\pi_i' = -\pi_i$. Con esta convención:

- La relación de potenciales para los arcos básicos se convierte en:
$$c_{ij} + \pi_i' - \pi_j' = 0 \quad \Longrightarrow \quad \pi_j' = \pi_i' + c_{ij}$$
lo que permite calcular todos los potenciales recorriendo el árbol desde el nodo raíz (con $\pi_r' = 0$) en sentido descendente.

- El costo reducido de un arco $(i,j)$ no básico es:
$$\bar{c}_{ij} = c_{ij} + \pi_i' - \pi_j'$$

- La condición de optimalidad se mantiene: $\bar{c}_{ij} \geq 0$ para todo $(i,j) \notin T$.

Esta convención da a $\pi_i'$ la interpretación de **precio de acceso al nodo $i$**: el costo de enviar una unidad de flujo desde el nodo raíz hasta el nodo $i$ a través del árbol actual.


### Ejemplo

Encuentre la asignación de flujos que minimiza el costo total de transporte en la siguiente red. Considere que la solución inicial es la entregada. 

![](../Images/ej_SR_1.png)

**Inicialización**

- Calcular los flujos en los arcos del árbol inicial resolviendo el sistema triangular $Bx_B = b$.
- Fijar $\pi_r' = 0$ en el nodo raíz y propagar los potenciales por el árbol usando $\pi_j' = \pi_i' + c_{ij}$ para cada arco $(i,j)$ básico.
- Calcular los costos reducidos de los arcos no básicos: $\bar{c}_{ij} = c_{ij} + \pi_i' - \pi_j'$.


![](../Images/ej_SR_2.png)

![](../Images/ej_SR_3.png)

**Iteración 1**

- Seleccionar el arco no básico con mayor costo reducido negativo (regla de Dantzig) o cualquier arco con $\bar{c}_{ij} < 0$.
- Agregar el arco $(p,q)$ a $T^{(k)}$ y determinar el ciclo $C$ que se forma.
- Calcular $\Delta = \min_{(i,j) \in C^-} x_{ij}$ y actualizar los flujos a lo largo del ciclo.
- El arco $(k,l)$ con $x_{kl} = 0$ sale del árbol.


![](../Images/ej_SR_4.png)

![](../Images/ej_SR_5.png)

**Iteración 2**

- Calcular los flujos en el nuevo árbol, actualizar los potenciales y los costos reducidos.
- Verificar si se cumple la condición de optimalidad.


![](../Images/ej_SR_6.png)

## Análisis de sensibilidad

Los potenciales $\pi_i'$ son los valores de las variables duales del problema de flujo de costo mínimo en la solución óptima. Permiten realizar análisis de sensibilidad sin necesidad de resolver el problema de nuevo:

- **Rango de optimalidad de un costo:** ¿Cuánto puede variar $c_{ij}$ para que la base óptima no cambie? El costo $c_{ij}$ puede variar sin cambiar de base mientras $\bar{c}_{ij} \geq 0$ para los arcos no básicos y mientras los potenciales no generen costos reducidos negativos en otros arcos.

Por ejemplo: ¿cuánto debería disminuir $c_{12}$ para que la solución actual deje de ser óptima?

![](../Images/ej_SR_7.png)


¿Qué rango de valores de $c_{32}$ mantienen la base óptima actual?

![](../Images/ej_SR_8.png)

![](../Images/ej_SR_9.png)


### Interpretación de los potenciales

Los potenciales $\pi_i'$ tienen una interpretación geoeconómica directa: representan el **costo marginal de suministro** en el nodo $i$, es decir, el costo mínimo de enviar una unidad adicional de flujo desde el nodo raíz hasta el nodo $i$ a través del árbol óptimo actual.

Esta interpretación es útil para el análisis de sensibilidad y para la toma de decisiones sobre la expansión de la red.

![](../Images/interpretacion_potenciales.png)


### Construcción de una solución factible inicial: Fase I

Para aplicar el algoritmo símplex de redes se necesita un árbol generador **factible** como punto de partida. Como la solución trivial $x = 0$ generalmente no es factible (viola las restricciones de balance), se utiliza el siguiente procedimiento auxiliar, análogo a la Fase I del símplex estándar:

1. Introducir un **nodo artificial** $s$ y conectarlo a cada nodo $i \in N$ con un arco artificial $(s, i)$ si $b_i < 0$ (nodo destino) o $(i, s)$ si $b_i > 0$ (nodo fuente).
2. Asignar a los arcos artificiales un costo $M \gg 0$ (método de la gran $M$) y a los arcos reales costo $0$.
3. El árbol inicial está formado por los $n-1$ arcos artificiales; los flujos iniciales son $x_{si} = -b_i$ o $x_{is} = b_i$ según corresponda.
4. Aplicar el símplex de redes hasta que todos los arcos artificiales salgan de la base.

**Teorema (factibilidad):** Un problema de flujo de costo mínimo admite solución factible si y solo si el valor óptimo de la Fase I es cero, es decir, si todos los arcos artificiales tienen flujo nulo en el óptimo de la Fase I.


![](../Images/fase_I_simplex.png)

![](../Images/FaseISimplex.png)


### Casos particulares

-  Así como una instancia del problema de flujo a costo mínimo puede ser infactible, existen otras situaciones especiales que podrían darse y que son convenientes de estudiar. A continuación, estudiaremos algunos casos particulares.


**Solución degenerada**

Una **solución básica factible degenerada** es aquella en la que al menos una variable básica tiene valor cero, es decir, existe un arco en el árbol con flujo nulo. La degeneración puede ocurrir cuando, al calcular $\Delta$, el mínimo se alcanza en dos o más arcos simultáneamente (empate en la variable que sale de la base).


Cuando existe un empate entre las variables del ciclo $C^-$ con igual valor de flujo mínimo, cualquiera de ellas puede elegirse como arco saliente. Sin importar cuál se elija, en el nuevo árbol existirá al menos una variable básica con flujo nulo (solución degenerada), lo que puede dar lugar a iteraciones sin mejora del objetivo.


![](../Images/casos_especiales0.png)
![](../Images/casos_especiales.png)

Escogemos el arco $(21)$ para salir de la base.

![](../Images/casos_especiales2.png)

La solución que presentamos, con una variable básica con flujo nulo, es una solución degenerada.

Sigamos aplicando el algoritmo simplex de redes.

![](../Images/casos_especiales30.png)
![](../Images/casos_especiales3.png)

![](../Images/casos_especiales4.png)



**Observaciones sobre la degeneración:**

1. Para declarar la optimalidad fue necesario explorar más de una base dentro del mismo vértice del dominio factible.
2. La solución óptima es degenerada: existe una variable básica con flujo cero.
3. Una solución degenerada implica que más de una base del símplex está asociada al mismo vértice del poliedro factible.
4. Lo anterior puede llevar a iterar entre bases del mismo vértice sin mejora del objetivo (**ciclado**). Para prevenir el ciclado existen reglas de pivote anti-ciclado (p.ej., la regla de Bland).
5. Aunque la solución primal sea la misma en varias iteraciones degeneradas, los potenciales $\pi_i'$ sí pueden cambiar. Esto refleja que el problema dual tiene múltiples soluciones óptimas cuando el primal es degenerado.


**Soluciones óptimas múltiples**

Cuando en el óptimo existe un arco no básico $(i,j)$ con costo reducido $\bar{c}_{ij} = 0$, introducirlo en la base no modifica el valor del objetivo pero produce una base óptima diferente. En este caso, el problema tiene **infinitas soluciones óptimas**.


Un arco no básico con $\bar{c}_{ij} = 0$ puede entrar a la base sin empeorar ni mejorar el objetivo. Al realizarse el intercambio se obtiene un árbol $T_1$ distinto de $T_0$ pero con el mismo costo óptimo $z^*$.


![](../Images/casos_especiales5.png)

Si hacemos entrar al arco $(31)$ a la base, obtenemos una solución factible distinta a la anterior, pero con el mismo valor de la función objetivo.

![](../Images/casos_especiales6.png)

**Observaciones sobre las soluciones óptimas múltiples:**

1. Sea $T_0$ el árbol (base) óptimo encontrado al terminar el algoritmo.
2. Si existe un arco $(i,j) \notin T_0$ con $\bar{c}_{ij} = 0$, entonces existe otro árbol óptimo $T_1$ que contiene a $(i,j)$.
3. $z(T_0) = z(T_1)$: ambos árboles tienen el mismo costo óptimo.
4. Cualquier combinación convexa de soluciones óptimas también es óptima: si $x_0$ y $x_1$ son soluciones óptimas, entonces $x^* = \lambda x_0 + (1-\lambda)x_1$, con $\lambda \in [0,1]$, también es óptima.
5. Puede ocurrir que $\bar{c}_{ij} = 0$ pero que no exista un árbol óptimo distinto que contenga a $(i,j)$; esto sucede cuando el ciclo formado al agregar $(i,j)$ tiene $\Delta = 0$ (caso degenerado sin cambio de base).
6. En general, si existen $k$ arcos no básicos con costo reducido nulo y bases óptimas asociadas $x_0, x_1, \ldots, x_k$, entonces toda combinación convexa de ellas es óptima:
$$x^* = \sum_{l=0}^{k} \lambda_l\, x_l \qquad \text{con} \quad \sum_{l=0}^{k} \lambda_l = 1, \quad \lambda_l \geq 0 \quad \forall\, l$$


**Solución no acotada**

![](../Images/casos_especiales7.png)

Un problema de flujo de costo mínimo factible tiene **solución no acotada** si y solo si la red contiene un **ciclo dirigido de costo negativo** $C \subseteq A$, es decir, un ciclo tal que:

$$\sum_{(i,j) \in C} c_{ij} < 0$$

En este caso, enviar flujo ilimitado a lo largo del ciclo reduce el costo de forma indefinida. Desde el punto de vista práctico, la existencia de un ciclo de costo negativo indica generalmente un error en el modelado del problema.


### Integralidad de las soluciones

En ocasiones el problema de flujo en redes exige que los flujos sean enteros. Esta propiedad no requiere resolución como un programa entero mixto, gracias al siguiente resultado:

**Teorema de integralidad (Ahuja et al., 1993, Cap. 3):** Si los parámetros $b_i$, $u_{ij}$ y $l_{ij}$ son números enteros, entonces el problema de flujo de costo mínimo posee al menos una solución óptima entera.

La razón es que la matriz de incidencia nodo-arco es **totalmente unimodular (TU)**: todas sus submatrices cuadradas no singulares tienen determinante $\pm 1$. Para una matriz TU:

- Las columnas poseen exactamente dos elementos distintos de cero: un $+1$ (nodo cola) y un $-1$ (nodo cabeza), con el resto igual a cero.
- Para cualquier vector entero $b$, el sistema $Ax = b,\, x \geq 0$ tiene solución básica entera.

**Consecuencia práctica:** El símplex de redes, partiendo de un árbol inicial con flujos enteros y costos enteros, producirá siempre flujos enteros en todas las iteraciones y en la solución óptima.


---

## Referencias

1. Ahuja, R. K., Magnanti, T. L. y Orlin, J. B. (1993). *Network Flows: Theory, Algorithms, and Applications*. Prentice-Hall. (Cap. 7) — Algoritmo símplex de redes, potenciales y análisis de sensibilidad.
2. Bazaraa, M. S., Jarvis, J. J. y Sherali, H. D. (1990). *Linear Programming and Network Flows* (2.ª ed.). Wiley. (pp. 505–527)
3. Bertsimas, D. y Tsitsiklis, N. J. (1997). *Introduction to Linear Optimization*. Athena Scientific. (Cap. 4) — Teoría de dualidad y costos reducidos.
4. Medaglia, A. (s.f.). *Flujo en Redes: Notas de Clase*. Departamento de Ingeniería Industrial, Universidad de los Andes.
